In [1]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_fact_expected_credit_loss
#
# Layer
# -----
# Gold Layer - Fact Tables
#
# Purpose
# -------
# Build the loan-level Expected Credit Loss fact table using
# loan exposure and internal rating data.
#
# Grain
# -----
# One row per loan.
#
# Output
# ------
# fact_expected_credit_loss
#
# Enterprise Concepts
# -------------------
# ✓ IFRS 9
# ✓ Expected Credit Loss
# ✓ PD / LGD / EAD
# ✓ Credit Risk Analytics
# ✓ Gold Fact Table
# ✓ Drill-through Analytics
# ============================================================

from pyspark.sql.functions import *
from datetime import datetime

loan_fact_table = "fact_loan_exposure"
rating_table = "silver_rating"

target_table = "fact_expected_credit_loss"
pipeline_name = "nb_build_fact_expected_credit_loss"

run_start_time = datetime.now()

print("ERIP Fact Expected Credit Loss Build Started")

StatementMeta(, f4884c35-e62a-4c8a-90a3-e3ae3299393b, 3, Finished, Available, Finished, False)

ERIP Fact Expected Credit Loss Build Started


In [2]:
# ============================================================
# SECTION 2 - READ SOURCE TABLES
# ============================================================

fact_loan_exposure = spark.table(loan_fact_table)
silver_rating = spark.table(rating_table)

print(f"Loan Exposure Rows : {fact_loan_exposure.count()}")
print(f"Rating Rows        : {silver_rating.count()}")

StatementMeta(, f4884c35-e62a-4c8a-90a3-e3ae3299393b, 4, Finished, Available, Finished, False)

Loan Exposure Rows : 5000
Rating Rows        : 1000


In [3]:
# ============================================================
# SECTION 3 - BUILD FACT EXPECTED CREDIT LOSS
# ============================================================

fact_expected_credit_loss = (
    fact_loan_exposure.alias("l")
    .join(
        silver_rating.select(
            "customer_id",
            "rating_sk",
            "current_internal_grade",
            "rating_risk_category",
            "pd",
            "lgd",
            "ead",
            "expected_loss",
            "model_version",
            "model_override_flag",
            "is_model_override",
            "ifrs9_stage_recommendation",
            "ifrs9_stage_numeric"
        ).alias("r"),
        "customer_id",
        "left"
    )
    .withColumn(
        "calculated_ecl",
        col("l.exposure_at_default") * col("r.pd") * col("r.lgd")
    )
    .withColumn(
        "ecl_variance",
        col("calculated_ecl") - col("r.expected_loss")
    )
    .withColumn(
        "ecl_risk_band",
        when(col("calculated_ecl") >= 5000000, "Very High ECL")
        .when(col("calculated_ecl") >= 1000000, "High ECL")
        .when(col("calculated_ecl") >= 250000, "Medium ECL")
        .otherwise("Low ECL")
    )
    .select(
        col("l.loan_sk"),
        col("l.loan_id"),
        col("l.facility_id"),
        col("l.customer_sk"),
        col("l.customer_id"),
        col("l.country_sk"),
        col("l.industry_sk"),
        col("r.rating_sk"),
        col("l.product_type"),
        col("l.facility_status"),
        col("l.exposure_at_default"),
        col("l.outstanding_balance"),
        col("l.approved_limit"),
        col("l.risk_weighted_assets"),
        col("r.current_internal_grade"),
        col("r.rating_risk_category"),
        col("r.pd"),
        col("r.lgd"),
        col("r.ead").alias("rating_ead"),
        col("r.expected_loss").alias("source_expected_loss"),
        col("calculated_ecl"),
        col("ecl_variance"),
        col("ecl_risk_band"),
        col("r.ifrs9_stage_recommendation"),
        col("r.ifrs9_stage_numeric"),
        col("r.model_version"),
        col("r.model_override_flag"),
        col("r.is_model_override"),
        current_timestamp().alias("gold_updated_timestamp")
    )
)

print(f"ECL fact rows created: {fact_expected_credit_loss.count()}")
display(fact_expected_credit_loss.limit(10))

StatementMeta(, f4884c35-e62a-4c8a-90a3-e3ae3299393b, 5, Finished, Available, Finished, False)

ECL fact rows created: 5000


SynapseWidget(Synapse.DataFrame, c107752c-16b1-495b-971e-1cf2f970fa40)

In [4]:
# ============================================================
# SECTION 4 - FACT ECL QUALITY VALIDATION
# ============================================================

total_rows = fact_expected_credit_loss.count()

duplicate_loans = total_rows - fact_expected_credit_loss.select("loan_id").distinct().count()

null_loan_sk = fact_expected_credit_loss.filter(col("loan_sk").isNull()).count()

null_customer_sk = fact_expected_credit_loss.filter(col("customer_sk").isNull()).count()

null_rating_sk = fact_expected_credit_loss.filter(col("rating_sk").isNull()).count()

invalid_pd = fact_expected_credit_loss.filter((col("pd") < 0) | (col("pd") > 1)).count()

invalid_lgd = fact_expected_credit_loss.filter((col("lgd") < 0) | (col("lgd") > 1)).count()

negative_ecl = fact_expected_credit_loss.filter(col("calculated_ecl") < 0).count()

print("Fact Expected Credit Loss Quality Checks")
print("---------------------------------------")
print(f"Rows               : {total_rows}")
print(f"Duplicate Loan IDs : {duplicate_loans}")
print(f"Null Loan SK       : {null_loan_sk}")
print(f"Null Customer SK   : {null_customer_sk}")
print(f"Null Rating SK     : {null_rating_sk}")
print(f"Invalid PD         : {invalid_pd}")
print(f"Invalid LGD        : {invalid_lgd}")
print(f"Negative ECL       : {negative_ecl}")

if (
    duplicate_loans > 0 or
    null_loan_sk > 0 or
    null_customer_sk > 0 or
    null_rating_sk > 0 or
    invalid_pd > 0 or
    invalid_lgd > 0 or
    negative_ecl > 0
):
    raise Exception("Fact Expected Credit Loss Validation Failed")
else:
    print("✓ Fact Expected Credit Loss Validation Passed")

StatementMeta(, f4884c35-e62a-4c8a-90a3-e3ae3299393b, 6, Finished, Available, Finished, False)

Fact Expected Credit Loss Quality Checks
---------------------------------------
Rows               : 5000
Duplicate Loan IDs : 0
Null Loan SK       : 0
Null Customer SK   : 0
Null Rating SK     : 0
Invalid PD         : 0
Invalid LGD        : 0
Negative ECL       : 0
✓ Fact Expected Credit Loss Validation Passed


In [5]:
# ============================================================
# SECTION 5 - WRITE GOLD FACT TABLE
# ============================================================

(
    fact_expected_credit_loss.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(target_table)
)

print(f"✓ Gold fact table created: {target_table}")
print(f"Rows written: {fact_expected_credit_loss.count()}")

StatementMeta(, f4884c35-e62a-4c8a-90a3-e3ae3299393b, 7, Finished, Available, Finished, False)

✓ Gold fact table created: fact_expected_credit_loss
Rows written: 5000
